# Assessment 5 Final Report (PDF Submission)

## Cover page

**Project:** Metabolic Risk Digital Twin (NHANES 2017–2018)  
**Student:** <Your Name>  
**Course / Module:** <Course Name>  
**Instructor:** <Professor Name>  
**Date:** 02 April 2026

---

## How to export this file to PDF

### Option A — VS Code (fastest)

1. Open this file in VS Code.
2. Open **Markdown Preview**.
3. Use **Print** from the preview (or browser print if it opens in a browser) → **Save as PDF**.

### Option B — Pandoc (if installed)

From repo root:

```bash
pandoc docs/assessment5/ASSESSMENT_5_FINAL_PDF.md -o Assessment_5_Report.pdf
```

If images do not render in your PDF, use Option A (VS Code preview) or paste screenshots into the Evidence Appendix.

---

## 1) Executive summary

This project builds a **Patient Digital Twin** for metabolic risk screening using **NHANES 2017–2018**. Each patient is represented as a clinical feature vector and the system produces:

- a **risk probability** in [0, 1]
- a **risk label** (0/1) from a threshold policy

The project is **reproducible and demo-ready**, with saved artifacts, automated tests, an API, and a React UI.

**Constraint respected:** existing research notebooks were not modified.

---

## 2) System architecture (high-level)

**Pipeline:** NHANES data → clean dataset → training/tuning → saved artifacts → API → UI.

If Mermaid diagrams do not render in your PDF export, take a screenshot of the diagram from the Markdown Preview and paste it into the Evidence Appendix.

```mermaid
flowchart LR
  A[Raw NHANES XPT files\n(data/raw)] --> B[Build / Merge / Clean\n(patient_state_clean.csv)]
  B --> C[Training + Tuning\n(scripts/train.py)]
  C --> D[Exported Artifacts\n(artifacts/*.joblib + *.metadata.json)]
  D --> E[FastAPI Inference Service\nGET /health\nPOST /predict]
  E --> F[React UI (Vite)\n(frontend)]
  D --> G[Offline CLI Inference\n(load joblib + predict_proba)]
  C --> H[Reports\nconfusion matrix\n(reports/figures)]
```

---

## 3) Dataset and target

- **Dataset:** NHANES 2017–2018
- **Join key:** `SEQN` (participant identifier)
- **Modeling dataset:** `data/processed/patient_state_clean.csv`

**Target label:** `Metabolic_Risk` is created from HbA1c thresholding used in the research notebooks.

- HbA1c is used to create the label only
- HbA1c is excluded from model inputs (leakage prevention)

---

## 4) Digital Twin state (features)

The deployed model uses 9 inputs (confirmed in artifact metadata):

- Age
- Gender (Male=0, Female=1)
- BMI
- Systolic_BP
- Diastolic_BP
- Glucose
- Insulin
- Total_Cholesterol
- HDL_Cholesterol

---

## 5) Modeling approach

### 5.1 Models explored (research notebooks)

- Logistic Regression
- Random Forest
- Support Vector Machine (RBF)
- Gradient Boosting

### 5.2 Deployed model (demo)

A scikit-learn Pipeline:

1. Median imputation
2. Standard scaling
3. Logistic Regression

---

## 6) Hyperparameter tuning + evaluation (artifact-ground-truth)

**Split:** stratified train/test, test_size=0.2, random_state=42  
**Tuning:** 5-fold stratified CV, optimized metric = F1

**Best params (saved):** `C=0.1`, `penalty=l2`  
**Best CV F1 (saved):** 0.7369406318

**Test metrics (saved):**
| Decision policy | Threshold | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| Default (`predict()`) | null | 0.75517 | 0.71296 | 0.65812 | 0.68444 |
| Screening-friendly | 0.4 | 0.75517 | 0.68110 | 0.73932 | 0.70902 |

Interpretation: threshold=0.4 improves recall and F1 in this run, aligning with screening goals.

---

## 7) Confusion matrices (embedded figures)

These images are generated by the training script and are stored in `reports/figures/`.

### 7.1 Threshold = 0.4

![](../../reports/figures/logreg_pipeline.confusion_matrix.png)

### 7.2 Default decision rule

![](../../reports/figures/logreg_pipeline_default.confusion_matrix.png)

---

## 8) Test Report (rigorous testing)

### 8.1 Automated tests (executed)

Command executed:

```bash
.venv/bin/python -m pytest -q
```

Outcome: `1 passed`.

### 8.2 Artifact reproducibility (executed)

Verified that `artifacts/logreg_pipeline.joblib` loads and produces `predict_proba()` output on a sample input (ensures export/reload works).

---

## 9) Validation & Verification

- Proper train/test separation (stratified split)
- Hyperparameter tuning via stratified 5-fold CV
- Metrics reported on holdout test set
- Threshold policy validated (default vs 0.4)
- Leakage prevention: HbA1c not used as model input
- API/UI integration supported via `/health`, `/predict`, and CORS checks

---

## 10) Demo checklist (short)

1. Start API: `.venv/bin/python -m uvicorn healthcare_digital_twin.api:app --reload --port 8000`
2. Check health: `curl http://127.0.0.1:8000/health`
3. Start UI: `cd frontend && npm run dev`
4. Predict in UI and explain probability + threshold

---

## 11) Evidence Appendix (paste screenshots here)

Paste screenshots into your final PDF if any embedded parts don’t render.

**A. Pytest passing (screenshot)**

**B. Artifact load + prediction output (screenshot)**

**C. API health response (screenshot)**

**D. UI prediction result (screenshot)**

---

## 12) Limitations

- NHANES is cross-sectional: model estimates risk, not clinical diagnosis
- External validation is required for real clinical deployment
- Threshold selection depends on the false-negative vs false-positive cost tradeoff
